# Merge and rename CDC SVI Files
Current CDC SVI files have input variables and SVI ranks for the US and Texas in different files.
This notebook merges the files and renames the variables to match the SVI.

## Description of Program
- program:    tu3svi2_2bv1_SetupCDC
- task:       Clean CDC files and rename variables
- Version:    2023-11-09
- 2023-11-30 - Add SVI Quartiles and Match
- 2024-08-27 - Update for 2020 ACS - keep all of US and TX
- 2024-09-06 - Issue with how many decimal places to keep in the SVI was keeping 1 but need all
- project:    DOE Southeast Texas Urban Field Lab SVI Paper - round 2 of analysis 
- funding:	  DOE
- author:    Nathanael Rosenheim
- GRA:        Lidia Mezei

## Control Python
Add install and add packages.

In [1]:
import pandas as pd     # For obtaining and cleaning tabular data
import geopandas as gpd # For obtaining and cleaning spatial data
import numpy as np      # For working with arrays
import os # For saving to path

In [2]:
import sys
print("Python Version     ", sys.version)
print("geopandas version: ", gpd.__version__)

Python Version      3.10.9 | packaged by conda-forge | (main, Feb  2 2023, 20:14:58) [MSC v.1929 64 bit (AMD64)]
geopandas version:  0.12.2


In [3]:
# Add the folder above the current notebook directory to the Python path
sys.path.append('../')

# Import the script as a module
import tu3svi2_0bv1_sourcedatautility_20240906 as my_utility

## Set Provenance 

In [4]:
# Get information on current working directory (getcwd)
os.getcwd()

'c:\\Users\\nathanael99\\Box\\SETx-UIFL_Team-folder\\Theme 3 - equity\\Text\\SVI_Analysis_Paper\\SourceData\\www_atsdr_cdc_gov_placeandhealth_svi'

In [5]:
# Store Program Name for output files to have the same name and saved in the same directory
programname = "tu3svi2_2bv2_SetupCDC_2024-09-06"
# Make directory to save output
#if not os.path.exists(programname):
#   os.mkdir(programname)

# Obtain Data
Obtain data from about each variable in SVI data.

In [6]:
# read in excel file
# Read in sheet with all metadata on each variable
varmetadata_df = pd.read_excel("../tu3svi2_0av3_SVIdatadictionary_2024-08-26.xlsx",
                            sheet_name="SVI_vars")
# sort by order
varmetadata_df = varmetadata_df.sort_values(by=['order'])
varmetadata_df[(varmetadata_df['SVI']=='CDC')].head()

,SVI,order,year,newvarname,gencat,gencatcode,comcat,comcatcode,theme,SVIcat,label,oldvarname,ACStable1v2,ACStable1v1,inverted,proportion,percent,dtype
0,CDC,10010,2018,C201810010,Geocode,100,TRACT2010,10,NaN,Geocode,Census Tract 11 Digit Geocode,FIPS,NaN,NaN,0,0,0,str
88,CDC,10010,2020,C202010010,Geocode,100,TRACT2020,10,NaN,Geocode,Census Tract 11 Digit Geocode,FIPS,NaN,NaN,0,0,0,str
89,CDC,10020,2020,C202010020,Denominator,100,Population,20,NaN,Denominator,Population,E_TOTPOP,S0601,S0601,0,0,0,int64
1,CDC,10020,2018,C201810020,Denominator,100,Population,20,NaN,Denominator,Population,E_TOTPOP,S0601,S0601,0,0,0,int64
90,CDC,10030,2020,C202010030,Denominator,100,Households,30,NaN,Denominator,Households,E_HH,DP02,DP02,0,0,0,int64


In [7]:
data_year = 2020
svi_name = 'CDC'
geoscales = ['Tract']  # census tract and block group
files_to_merge = {  'TractTX' : {'folder' : '', 
                                'inputvars' : f'CDC_{data_year}_TX_Tract_2023-11-09.csv'},
                    'TractUS' : {'folder' : '', 
                                'inputvars' : f'CDC_{data_year}_US_Tract_2023-11-09.csv'}
                }

### Merge plan
1. Merge tract TX and US files

This will go from 2 files to 1 file. The TX and US have the same input variables but will have different SVI ranks.

In [8]:
merge_files = {}
for mergename in files_to_merge:
    print(mergename)
    # Set path to folder containing the files to be merged
    path = files_to_merge[mergename]['folder']
    # Set path to input variables file
    inputvars = files_to_merge[mergename]['inputvars']
    
    # Read in input variables file
    df_inputvars = pd.read_csv(inputvars, dtype={'FIPS' : str})
   
    # Save merged file to shapefile
    merge_files[mergename] = df_inputvars

TractTX
TractUS


In [9]:
merge_files['TractUS'].head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,1,Alabama,AL,1001,Autauga,01001020100,"Census Tract 201, Autauga County, Alabama",3.793570,1941,390,...,2.1,2.7,0.0,1.8,0.0,1.8,6.6,5.1,0.0,1.8
1,1,Alabama,AL,1001,Autauga,01001020200,"Census Tract 202, Autauga County, Alabama",1.282174,1757,310,...,0.0,2.0,0.0,2.0,0.0,2.0,2.6,3.0,0.8,1.1
2,1,Alabama,AL,1001,Autauga,01001020300,"Census Tract 203, Autauga County, Alabama",2.065364,3694,570,...,1.2,1.1,0.0,0.9,0.0,0.9,1.8,2.3,0.0,0.9
3,1,Alabama,AL,1001,Autauga,01001020400,"Census Tract 204, Autauga County, Alabama",2.464984,3539,500,...,0.5,0.6,0.3,0.5,0.0,1.0,2.9,2.8,0.0,1.0
4,1,Alabama,AL,1001,Autauga,01001020501,"Census Tract 205.01, Autauga County, Alabama",2.395243,4306,662,...,1.9,2.0,0.0,0.8,0.0,0.8,0.3,0.6,0.0,0.8


In [10]:
merge_files['TractTX'].head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,48,Texas,TX,48001,Anderson,48001950100,"Census Tract 9501, Anderson County, Texas",186.605650,4958,736,...,1.0,1.4,0.4,0.5,0.0,0.9,8.0,4.7,0.0,0.1
1,48,Texas,TX,48001,Anderson,48001950401,"Census Tract 9504.01, Anderson County, Texas",6.374261,4867,838,...,0.2,0.3,0.2,0.4,0.0,0.9,2.2,1.5,0.3,0.4
2,48,Texas,TX,48001,Anderson,48001950402,"Census Tract 9504.02, Anderson County, Texas",27.465086,7335,352,...,0.1,0.2,0.1,0.2,0.1,0.2,1.1,0.8,0.0,0.6
3,48,Texas,TX,48001,Anderson,48001950500,"Census Tract 9505, Anderson County, Texas",8.931332,4397,575,...,0.0,1.0,0.0,1.0,0.0,1.0,1.8,1.5,0.0,1.0
4,48,Texas,TX,48001,Anderson,48001950600,"Census Tract 9506, Anderson County, Texas",7.974505,4704,915,...,0.0,1.0,0.0,1.0,0.0,1.0,1.5,2.3,0.0,1.0


## Add SVI without Minority
The CDRZ designations do not use minority. Therefore need to have an extra SVI without minority.

In [11]:
for keys in merge_files:
    # replace -9999 with missing
    merge_files[keys].replace(-999, np.nan, inplace=True)

    # SPL_THEME without minority subtracts EPL_MINRTY from SPL_THEMES
    merge_files[keys]['SPL_THEMES_v2'] = merge_files[keys]['SPL_THEMES'] - merge_files[keys]['EPL_MINRTY']
    # calculate percentile rank
    merge_files[keys]['RPL_THEMES_v2'] = merge_files[keys]['SPL_THEMES_v2'].rank(pct=True)
    # keep four decimal places
    merge_files[keys]['RPL_THEMES_v2'] = merge_files[keys]['RPL_THEMES_v2'].apply(lambda x: round(x, 4))
    # drop SPL_THEMES_v2
    merge_files[keys].drop(columns=['SPL_THEMES_v2'], inplace=True)

In [12]:
check_vars = ['SPL_THEMES','EPL_MINRTY','RPL_THEMES','RPL_THEMES_v2']
merge_files['TractUS'][check_vars].describe().T

,count,mean,std,min,25%,50%,75%,max
SPL_THEMES,83331.0,7.674277,2.254260,0.0,6.0034,7.6242,9.36505,14.3200
EPL_MINRTY,83598.0,0.499376,0.288914,0.0,0.2496,0.4994,0.74970,0.9959
RPL_THEMES,83331.0,0.499994,0.288681,0.0,0.2500,0.5000,0.75000,1.0000
RPL_THEMES_v2,83331.0,0.500006,0.288677,0.0,0.2500,0.5000,0.75000,1.0000


## Add SVI using E_HBURD instead of EP_HBURD

https://www.atsdr.cdc.gov/placeandhealth/svi/data_documentation_download.html

CDC Data Correction
Data values for Housing Burden and Overall SVI in the 2020 SVI dataset have been updated. These variables were previously using E_HBURD instead of EP_HBURD in the calculation and thus the values after release on 10/27/22 and before 12/22/22 were incorrect.

In [13]:
check_vars = ['SPL_THEMES','RPL_THEMES','E_HBURD','EP_HBURD','EPL_HBURD']
merge_files['TractUS'][check_vars].describe().T

,count,mean,std,min,25%,50%,75%,max
SPL_THEMES,83331.0,7.674277,2.254260,0.0,6.0034,7.6242,9.36505,14.3200
RPL_THEMES,83331.0,0.499994,0.288681,0.0,0.2500,0.5000,0.75000,1.0000
E_HBURD,84122.0,395.579361,241.865155,0.0,222.0000,349.0000,519.00000,4712.0000
EP_HBURD,84122.0,27.229401,12.787458,0.0,18.0000,25.2000,34.80000,100.0000
EPL_HBURD,83598.0,0.498775,0.288946,0.0,0.2470,0.4995,0.74860,0.9996


In [14]:
for keys in merge_files:
    # replace -9999 with missing
    merge_files[keys].replace(-999, np.nan, inplace=True)

    # calculate percentile rank for housing burden using E_HBURD
    merge_files[keys]['EPL_HBURD_v2'] = merge_files[keys]['E_HBURD'].rank(pct=True)

    # SPL_THEME without minority subtracts EPL_MINRTY from SPL_THEMES
    merge_files[keys]['SPL_THEMES_v3'] = merge_files[keys]['SPL_THEMES'] - merge_files[keys]['EPL_HBURD'] + merge_files[keys]['EPL_HBURD_v2']
    # calculate percentile rank
    merge_files[keys]['RPL_THEMES_v3'] = merge_files[keys]['SPL_THEMES_v3'].rank(pct=True)
    # keep four decimal places
    merge_files[keys]['RPL_THEMES_v3'] = merge_files[keys]['RPL_THEMES_v3'].apply(lambda x: round(x, 4))
    # drop SPL_THEMES_v2
    merge_files[keys].drop(columns=['SPL_THEMES_v3'], inplace=True)

In [15]:
check_vars = ['SPL_THEMES','RPL_THEMES','RPL_THEMES_v3','E_HBURD','EP_HBURD','EPL_HBURD','EPL_HBURD_v2']
merge_files['TractUS'][check_vars].describe().T

,count,mean,std,min,25%,50%,75%,max
SPL_THEMES,83331.0,7.674277,2.254260,0.000000,6.003400,7.624200,9.365050,14.3200
RPL_THEMES,83331.0,0.499994,0.288681,0.000000,0.250000,0.500000,0.750000,1.0000
RPL_THEMES_v3,83331.0,0.500006,0.288677,0.000000,0.250000,0.500000,0.750000,1.0000
E_HBURD,84122.0,395.579361,241.865155,0.000000,222.000000,349.000000,519.000000,4712.0000
EP_HBURD,84122.0,27.229401,12.787458,0.000000,18.000000,25.200000,34.800000,100.0000
EPL_HBURD,83598.0,0.498775,0.288946,0.000000,0.247000,0.499500,0.748600,0.9996
EPL_HBURD_v2,84122.0,0.500006,0.288676,0.005486,0.249067,0.499733,0.749626,1.0000


## Merge datafiles to have one set for Tract but with SVI_scaled by US and State

In [16]:
# start dictionary to save merged files
merge_by_geo = {}
for geoscale in geoscales:
    # merge merged_files['BGSETX'] with merged_files['BGTX']
    df1 = merge_files[geoscale+'TX'].copy(deep=True)
    df2 = merge_files[geoscale+'US'].copy(deep=True)
    keep_vars = ['FIPS','RPL_THEMES','RPL_THEMES_v2','RPL_THEMES_v3','F_TOTAL']

    # rename RPL_THEMES to RPL_THEMES_US
    df1 = df1[keep_vars].copy(deep=True)
    df2 = df2.copy(deep=True)
    df2 = df2.rename(columns={'RPL_THEMES' : 'RPL_THEMES_US'})
    df1 = df1.rename(columns={'RPL_THEMES' : 'RPL_THEMES_TX'})
    df2 = df2.rename(columns={'RPL_THEMES_v2' : 'RPL_THEMES_v2_US'})
    df1 = df1.rename(columns={'RPL_THEMES_v2' : 'RPL_THEMES_v2_TX'})
    df2 = df2.rename(columns={'RPL_THEMES_v3' : 'RPL_THEMES_v3_US'})
    df1 = df1.rename(columns={'RPL_THEMES_v3' : 'RPL_THEMES_v3_TX'})
    df2 = df2.rename(columns={'F_TOTAL' : 'F_TOTAL_US'})
    df1 = df1.rename(columns={'F_TOTAL' : 'F_TOTAL_TX'})

    # replace missing values with nan
    df1 = df1.replace(-999, np.nan)
    df2 = df2.replace(-999, np.nan)
    
    # sort by FIPS
    df1 = df1.sort_values(by=['FIPS'])
    df2 = df2.sort_values(by=['FIPS'])
    # merge df1 and df2 - keep all of the US
    merge_by_geo[geoscale] = pd.merge(df1, df2, on='FIPS', how='right')



In [17]:
merge_by_geo['Tract'].head()

,FIPS,RPL_THEMES_TX,RPL_THEMES_v2_TX,RPL_THEMES_v3_TX,F_TOTAL_TX,ST,STATE,ST_ABBR,STCNTY,COUNTY,...,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE,RPL_THEMES_v2_US,EPL_HBURD_v2,RPL_THEMES_v3_US
0,01001020100,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,1.8,0.0,1.8,6.6,5.1,0.0,1.8,0.2786,0.108248,0.2397
1,01001020200,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,2.0,0.0,2.0,2.6,3.0,0.8,1.1,0.4885,0.115861,0.4721
2,01001020300,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,0.9,0.0,0.9,1.8,2.3,0.0,0.9,0.4844,0.333932,0.5055
3,01001020400,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,0.5,0.0,1.0,2.9,2.8,0.0,1.0,0.2897,0.374331,0.2836
4,01001020501,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,0.8,0.0,0.8,0.3,0.6,0.0,0.8,0.3431,0.503542,0.3514


In [18]:
# look RPL_THEME columns
RPL_THEME_cols = [col for col in merge_by_geo['Tract'].columns if 'RPL_THEMES' in col]
merge_by_geo['Tract'][RPL_THEME_cols].head()

,RPL_THEMES_TX,RPL_THEMES_v2_TX,RPL_THEMES_v3_TX,RPL_THEMES_US,RPL_THEMES_v2_US,RPL_THEMES_v3_US
0,NaN,NaN,NaN,0.2823,0.2786,0.2397
1,NaN,NaN,NaN,0.5406,0.4885,0.4721
2,NaN,NaN,NaN,0.5042,0.4844,0.5055
3,NaN,NaN,NaN,0.2703,0.2897,0.2836
4,NaN,NaN,NaN,0.3343,0.3431,0.3514


### Add SVI Quartiles and Match

In [19]:
merge_by_geo['Tract']['FIPS'].describe()

count           84122
unique          84122
top       01001020100
freq                1
Name: FIPS, dtype: object

In [20]:
for file in ['Tract']:
    merge_by_geo[file] = \
        my_utility.quartileSVI(merge_by_geo[file], 
                               'RPL_THEMES_US', 'RPL_THEMES_US')
    merge_by_geo[file] = \
        my_utility.quartileSVI(merge_by_geo[file], 
                               'RPL_THEMES_TX', 'RPL_THEMES_TX')

    # Filling NA values with a placeholder (for example, a string 'NA') before comparison
    merge_by_geo[file]['RPL_THEMES_US_sviq'] = \
        merge_by_geo[file]['RPL_THEMES_US_sviq'].astype(str).fillna('NA')
    merge_by_geo[file]['RPL_THEMES_TX_sviq'] = \
        merge_by_geo[file]['RPL_THEMES_TX_sviq'].astype(str).fillna('NA')

    # create a new binary variable to check if SVIr_sviq and SVIp_sviq match
    merge_by_geo[file]['sviq_match'] = \
        np.where(merge_by_geo[file]['RPL_THEMES_US_sviq'] == \
                 merge_by_geo[file]['RPL_THEMES_TX_sviq'], 1, 0)
    # describe new variable
    merge_by_geo[file]['sviq_match'].describe()

    # replace svi_match with missing if SVI_scaled_TX_sviq or SVI_scaled_SETX_sviq is NA
    merge_by_geo[file]['sviq_match'] = \
        np.where(merge_by_geo[file]['RPL_THEMES_US_sviq'] == \
                 "<NA>", np.nan, merge_by_geo[file]['sviq_match'])
    merge_by_geo[file]['sviq_match'] = \
        np.where(merge_by_geo[file]['RPL_THEMES_TX_sviq'] == \
                 "<NA>", np.nan, merge_by_geo[file]['sviq_match'])

In [21]:
merge_by_geo['Tract']['sviq_match'].describe()

count    6827.000000
mean        0.671305
std         0.469773
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: sviq_match, dtype: float64

In [22]:
merge_by_geo['Tract'].head()

,FIPS,RPL_THEMES_TX,RPL_THEMES_v2_TX,RPL_THEMES_v3_TX,F_TOTAL_TX,ST,STATE,ST_ABBR,STCNTY,COUNTY,...,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE,RPL_THEMES_v2_US,EPL_HBURD_v2,RPL_THEMES_v3_US,RPL_THEMES_US_sviq,RPL_THEMES_TX_sviq,sviq_match
0,01001020100,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,6.6,5.1,0.0,1.8,0.2786,0.108248,0.2397,2,<NA>,NaN
1,01001020200,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,2.6,3.0,0.8,1.1,0.4885,0.115861,0.4721,3,<NA>,NaN
2,01001020300,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,1.8,2.3,0.0,0.9,0.4844,0.333932,0.5055,3,<NA>,NaN
3,01001020400,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,2.9,2.8,0.0,1.0,0.2897,0.374331,0.2836,2,<NA>,NaN
4,01001020501,NaN,NaN,NaN,NaN,1,Alabama,AL,1001,Autauga,...,0.3,0.6,0.0,0.8,0.3431,0.503542,0.3514,2,<NA>,NaN


# Rename variables
Using the Variable Data Dictionary Excel Sheet rename the variables to match across SVI.

In [23]:
varmetadata_df.head(1)

,SVI,order,year,newvarname,gencat,gencatcode,comcat,comcatcode,theme,SVIcat,label,oldvarname,ACStable1v2,ACStable1v1,inverted,proportion,percent,dtype
0,CDC,10010,2018,C201810010,Geocode,100,TRACT2010,10,NaN,Geocode,Census Tract 11 Digit Geocode,FIPS,NaN,NaN,0,0,0,str


In [24]:
geoscales

['Tract']

In [25]:
# find RPL columns
RPL_cols = [col for col in merge_by_geo['Tract'].columns if 'RPL' in col]
merge_by_geo['Tract'][RPL_cols].head()

,RPL_THEMES_TX,RPL_THEMES_v2_TX,RPL_THEMES_v3_TX,RPL_THEME1,RPL_THEME2,RPL_THEME3,RPL_THEME4,RPL_THEMES_US,RPL_THEMES_v2_US,RPL_THEMES_v3_US,RPL_THEMES_US_sviq,RPL_THEMES_TX_sviq
0,NaN,NaN,NaN,0.4578,0.5079,0.3921,0.0945,0.2823,0.2786,0.2397,2,<NA>
1,NaN,NaN,NaN,0.5348,0.0810,0.7610,0.7915,0.5406,0.4885,0.4721,3,<NA>
2,NaN,NaN,NaN,0.3639,0.8632,0.5529,0.3500,0.5042,0.4844,0.5055,3,<NA>
3,NaN,NaN,NaN,0.2081,0.8131,0.2386,0.1759,0.2703,0.2897,0.2836,2,<NA>
4,NaN,NaN,NaN,0.2540,0.6297,0.3313,0.3484,0.3343,0.3431,0.3514,2,<NA>


In [26]:
cleaned_files = my_utility.update_varnames(varmetadata_df= varmetadata_df,
                    svi_name = svi_name,
                    data_year = data_year,
                    merge_by_geo = merge_by_geo,
                    geoscales = geoscales,
                    programname = programname)

Current variable name:  FIPS
Found variable in variable dictionary
Then new variable name is:  TRACT2020 check length:  9
renaming FIPS to TRACT2020
Current variable name:  E_TOTPOP
Found variable in variable dictionary
Then new variable name is:  C202010020 check length:  10
renaming E_TOTPOP to C202010020
Current variable name:  E_HH
Found variable in variable dictionary
Then new variable name is:  C202010030 check length:  10
renaming E_HH to C202010030
Current variable name:  E_HU
Found variable in variable dictionary
Then new variable name is:  C202010040 check length:  10
renaming E_HU to C202010040
Current variable name:  EP_POV150
Found variable in variable dictionary
Then new variable name is:  C202020010 check length:  10
renaming EP_POV150 to C202020010
Current variable name:  EP_UNEMP
Found variable in variable dictionary
Then new variable name is:  C202020020 check length:  10
renaming EP_UNEMP to C202020020
Current variable name:  EP_HBURD
Found variable in variable dicti

In [27]:
cleaned_files['Tract'].head()

,TRACT2020,C202010020,C202010030,C202010040,C202020010,C202020020,C202020031,C202020050,C202021010,C202030010,...,C202090012,C202090021,C202090022,C202090031,C202090032,C202090091,C202090092,C202091011,C202091012,C202092023
0,01001020100,1941,693,710,0.181,0.021,0.208,0.096,0.143,0.152,...,NaN,0.2786,NaN,0.2397,NaN,0.0,NaN,2,<NA>,NaN
1,01001020200,1757,573,720,0.254,0.040,0.260,0.059,0.106,0.162,...,NaN,0.4885,NaN,0.4721,NaN,1.0,NaN,3,<NA>,NaN
2,01001020300,3694,1351,1464,0.228,0.027,0.195,0.035,0.128,0.126,...,NaN,0.4844,NaN,0.5055,NaN,0.0,NaN,3,<NA>,NaN
3,01001020400,3539,1636,1741,0.142,0.024,0.174,0.048,0.062,0.274,...,NaN,0.2897,NaN,0.2836,NaN,1.0,NaN,2,<NA>,NaN
4,01001020501,4306,1706,1786,0.210,0.010,0.206,0.032,0.088,0.126,...,NaN,0.3431,NaN,0.3514,NaN,0.0,NaN,2,<NA>,NaN


In [28]:
# find C2020900 cols
C2020900_cols = [col for col in cleaned_files['Tract'].columns if 'C2020900' in col]
cleaned_files['Tract'][C2020900_cols].head()

,C202090011,C202090012,C202090021,C202090022,C202090031,C202090032,C202090091,C202090092
0,0.2823,NaN,0.2786,NaN,0.2397,NaN,0.0,NaN
1,0.5406,NaN,0.4885,NaN,0.4721,NaN,1.0,NaN
2,0.5042,NaN,0.4844,NaN,0.5055,NaN,0.0,NaN
3,0.2703,NaN,0.2897,NaN,0.2836,NaN,1.0,NaN
4,0.3343,NaN,0.3431,NaN,0.3514,NaN,0.0,NaN


In [29]:
cleaned_files['Tract']['TRACT2020'].describe()

count           84122
unique          84122
top       01001020100
freq                1
Name: TRACT2020, dtype: object